# Predicting CANDOR survey responses based on growth rates $\alpha$

Calling my shot: my prediction here is that closer matched rates of change are related to developing joint perspective.

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# stats in R
import rpy2.robjects as ro
from rpy2.robjects import Formula
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri, default_converter, conversion

import warnings; warnings.filterwarnings('ignore')

In [2]:
var = 'developed_joint_perspective_sr2'

Data paths used in the project

In [3]:
SLOPES_PATH = '../../ComplexityScienceOfEverydayConvos/english/4-complexity_tests/allan-variance/av-params.parquet'

candor_meta_data_path = '../../ComplexityScienceOfEverydayConvos/english/data/all-candor-meta_data.csv'

TABLES_FIGS_PATH = 'tables_and_figures/'
if not os.path.exists(TABLES_FIGS_PATH):
    os.mkdir(TABLES_FIGS_PATH)

PARAMETER_ESTIMATES_PATH = 'params/'
if not os.path.exists(PARAMETER_ESTIMATES_PATH):
    os.mkdir(PARAMETER_ESTIMATES_PATH)

if not os.path.exists(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats')):
    os.mkdir(os.path.join(TABLES_FIGS_PATH, 'individual_var_stats'))

## Importing data

In [4]:
results = pd.read_parquet(SLOPES_PATH)
results = results.loc[~results['null']]
results = results.sort_values(by=['x_turn_id'])
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope,b0
0,CABNC-0missing-CABNC-0missing-KB0RE004-10,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.183603,-11.122748
1,CABNC-0missing-CABNC-0missing-KB0RE004-11,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.979970,-6.513821
2,CABNC-0missing-CABNC-0missing-KB0RE004-12,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-1.234872,-5.934323
3,CABNC-0missing-CABNC-0missing-KB0RE004-13,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,0.019517,-10.000541
4,CABNC-0missing-CABNC-0missing-KB0RE004-15,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-KB0PSUN,-0.995607,-9.275234


In [5]:
results = pd.merge(
    left=results.loc[results['self']], left_on=['x_turn_id'],
    right=results[['x_turn_id', 'slope']].loc[~results['self']], right_on=['x_turn_id'],
    how='left'
)
print(results.shape)
results.isna().sum()

(923235, 8)


x_turn_id              0
null                   0
self                   0
conversation_id        0
speaker                0
slope_x             2270
b0                  2256
slope_y            29271
dtype: int64

In [6]:
results['delta_slope'] = (results['slope_x'] - results['slope_y']).abs()

In [7]:
results = results.loc[~results['delta_slope'].isna()]
print(results.shape)
results.head()

(891343, 9)


,x_turn_id,null,self,conversation_id,speaker,slope_x,b0,slope_y,delta_slope
16,CABNC-0missing-CABNC-0missing-KB0RE004-6,False,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-PS009,4.500377,-9.520385,0.937405,3.562972
17,CABNC-0missing-CABNC-0missing-KB0RE006-16,False,True,CABNC-0missing-CABNC-0missing-KB0RE006,CABNC-0missing-CABNC-0missing-KB0RE006-PS007,-1.977476,-2.756554,1.212063,3.189539
21,CABNC-0missing-CABNC-0missing-KB1RE005-10,False,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01A,-0.616680,-9.841162,-1.021401,0.404721
22,CABNC-0missing-CABNC-0missing-KB1RE005-100,False,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01F,-1.206114,-6.915075,0.009618,1.215732
23,CABNC-0missing-CABNC-0missing-KB1RE005-101,False,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01A,0.142933,-11.794718,-0.972375,1.115308


##### Some preliminary statistics

In [ ]:
print((results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean())
results.isna().sum()

In [ ]:
results.isna().mean()

In [ ]:
(results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean()

In [ ]:
results['self'].value_counts()

In [ ]:
results.head()

### Adding Corpus names back in . . .

In [8]:
subcorpora_col = 'corpus'

In [9]:
# to rename the corpus correctly . . .
def lang(x):
    result = str(x)

    if '-DISPEL-' in result:
        result = 'DISPEL'

    if result.startswith('GCSAusE'):
        result = 'GCSAusE'

    if result.startswith('se'):
        result = 'CORAAL'

    if result.startswith('call'):
        result = x.split('-')[0]

    if result.startswith('MICASE'):
        result = 'MICASE'

    if result.startswith('instruction'):
        result = x.split('-')[1]

    if result.startswith('CABNC'):
        result = 'CABNC'

    if result.startswith('SBCSAE'):
        result = 'SBCSAE'

    if result.startswith('SCoSE'):
        result = 'SCoSE'

    if 'candor' in result.lower():
        result = 'CANDOR'

    if result.startswith('grace'):
        result = 'Miao-fNIRS'

    return result

In [10]:
results[subcorpora_col] = [lang(x) for x in tqdm(results['conversation_id'])]
# results[subcorpora_col] = [context(x) for x in tqdm(results['conversation_id'])]
# results.head()

  0%|          | 0/891343 [00:00<?, ?it/s]

In [11]:
# results[subcorpora_col].value_counts()

### Removing null-test results/bad values

In [12]:
sel = results['delta_slope'].isna() | (results['delta_slope'].abs() == np.inf)
# sel.sum(), sel.mean(), (~sel).sum(), results['x_turn_id'].loc[sel].nunique()

In [13]:
# is_null = np.array(['null-' in x for x in tqdm(results['x_turn_id'])])
is_null = results['null']
# is_null.sum()

In [14]:
results = results.loc[
    (~is_null)
    & (~sel)
    & results[subcorpora_col].isin(['CANDOR'])
]
results.shape

(388879, 10)

In [15]:
results['conversation_id'] = ['-'.join(it.split('-')[2:]) for it in tqdm(results['conversation_id'])]

  0%|          | 0/388879 [00:00<?, ?it/s]

In [16]:
results.head()

,x_turn_id,null,self,conversation_id,speaker,slope_x,b0,slope_y,delta_slope,corpus
148345,CANDOR-0-06a5a69a-362d-4947-b16d-d1d09dc075ec-10,False,True,06a5a69a-362d-4947-b16d-d1d09dc075ec,5c5c76a0bfe5280001447999,-0.478398,-6.809024,0.143927,0.622325,CANDOR
148346,CANDOR-0-06a5a69a-362d-4947-b16d-d1d09dc075ec-100,False,True,06a5a69a-362d-4947-b16d-d1d09dc075ec,5c5c76a0bfe5280001447999,-0.163754,-3.402271,-0.017286,0.146468,CANDOR
148347,CANDOR-0-06a5a69a-362d-4947-b16d-d1d09dc075ec-101,False,True,06a5a69a-362d-4947-b16d-d1d09dc075ec,5de40feb61872d000d8803ff,-0.065748,-10.131036,2.372082,2.437830,CANDOR
148348,CANDOR-0-06a5a69a-362d-4947-b16d-d1d09dc075ec-102,False,True,06a5a69a-362d-4947-b16d-d1d09dc075ec,5c5c76a0bfe5280001447999,0.222705,-7.384038,0.672415,0.449710,CANDOR
148350,CANDOR-0-06a5a69a-362d-4947-b16d-d1d09dc075ec-104,False,True,06a5a69a-362d-4947-b16d-d1d09dc075ec,5c5c76a0bfe5280001447999,-0.755032,-6.248620,0.212503,0.967535,CANDOR


## Merging with CANDOR metadata

In [17]:
dfm = pd.read_csv(candor_meta_data_path)[['user_id', 'convo_id', var]]
print(dfm.shape)
dfm.head()

(3312, 3)


,user_id,convo_id,developed_joint_perspective_sr2
0,5efbe4f0c9f37c1a3b70afaf,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,7.0
1,5f26378609aca228dcfd19fa,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,5.0
2,5d8e361e6e920e0014b459d9,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,1.0
3,5f4d435c030fef9d0136b496,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,2.0
4,5ca6bbf13b5fcf00100996e9,4e3233b7-5d89-47e9-a795-25452f99a1c9,3.0


In [18]:
dfm[list(dfm)[-1]].max(), dfm[list(dfm)[-1]].median(), dfm[list(dfm)[-1]].mean(), dfm[list(dfm)[-1]].min()

(np.float64(7.0),
 np.float64(5.0),
 np.float64(4.920454545454546),
 np.float64(1.0))

In [19]:
dfm[list(dfm)[-1]].mode()

0    5.0
Name: developed_joint_perspective_sr2, dtype: float64

In [20]:
dfm[list(dfm)[-1]].value_counts() / dfm[list(dfm)[-1]].value_counts().sum()

developed_joint_perspective_sr2
5.0    0.315418
6.0    0.257678
4.0    0.151413
7.0    0.118550
3.0    0.078624
2.0    0.047604
1.0    0.030713
Name: count, dtype: float64

test for (1) normalcy, and (2) uniformity

In [21]:
from scipy.stats import skewtest, chisquare
v = dfm.loc[~dfm[list(dfm)[-1]].isna()]
pv = v[list(dfm)[-1]].value_counts().values
print(skewtest(v[list(dfm)[-1]].values))
chisquare(f_obs=pv, f_exp=np.array([1/len(pv)]*len(pv))*pv.sum())

SkewtestResult(statistic=np.float64(-15.804049229104656), pvalue=np.float64(2.9175879730923192e-56))


Power_divergenceResult(statistic=np.float64(1581.7739557739558), pvalue=np.float64(0.0))

In [22]:
results = pd.merge(
    left=results, left_on=['speaker', 'conversation_id'],
    right=dfm, right_on=['user_id', 'convo_id'],
    # left=results, left_on=['speaker'],
    # right=dfm, right_on=['user_id'],
    how='left'
)

In [23]:
results = results.drop(columns=['user_id', 'convo_id'])
# results = results.drop(columns=['user_id'])
# results.columns
results.isna().sum()

x_turn_id                             0
null                                  0
self                                  0
conversation_id                       0
speaker                               0
slope_x                               0
b0                                    0
slope_y                               0
delta_slope                           0
corpus                                0
developed_joint_perspective_sr2    6830
dtype: int64

In [24]:
sel = results[[var]].isna().astype(int).sum(axis=1) == 0
sel.sum(), results.shape

(np.int64(382049), (388879, 11))

## Predicting individual survey variables from decay rates $\alpha$

### Converting categorical cols to numbered indexes

In [25]:
## convert speaker to integers
col = 'speaker'
vs = results[col].unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col])]

  0%|          | 0/388879 [00:00<?, ?it/s]

In [26]:
## convert turn_ids to integers
col = 'conversation_id'
vs = results[col].astype(str).unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
results[col+'_'] = [sid[sp] for sp in tqdm(results[col].astype(str))]

  0%|          | 0/388879 [00:00<?, ?it/s]

### Creating Model Script

In [27]:
# model_script = """
# model{
#     b0 ~ dunif(-100, 100)
#     beta ~ dunif(-100, 100)
#     phi ~ dunif(-100, 100)
#
#     for (p in 1:KSPEAKER){
#         sigma[p] ~ dunif(.0001, 10)
#     }
#
#     for (r in 1:ROWS){
#         mu[r] <- survey_var[r] * (beta + (phi * self_other_comp[r]))
#         slope[r] ~ dnorm(b0 + mu[r], 1/pow(sigma[speaker[r]],2))
#     }
# }
# """
# f = open('model.txt', 'w')
# f.write(model_script)
# f.close()

In [28]:
# model_script = """
# model{
#     b0 ~ dunif(-100, 100)
#     beta ~ dunif(-100, 100)
#     phi ~ dunif(-100, 100)
#     gamma ~ dunif(1, 10)
#     eta ~ dunif(1, 10)
#     omega ~ dunif(1, 10)
#
#     for (c in 1:CCONVERSATION){
#         conversation_sigma[c] ~ dnorm(omega, 1)
#     }
#
#     for (p in 1:KSPEAKER){
#         speaker_sigma[p] ~ dnorm(gamma, 1)
#         other_sigma[p] ~ dnorm(eta, 1)
#     }
#
#     for (r in 1:ROWS){
#         mu[r] <- survey_var[r] * (beta + (phi * self_other_comp[r]))
#         sigma[r] <- conversation_sigma[conversation[r]] + ifelse(self_other_comp[r] > 0, speaker_sigma[speaker[r]], other_sigma[speaker[r]])
#         slope[r] ~ dnorm(b0 + mu[r], 1/pow(sigma[r],2))
#     }
# }
# """
# f = open('model.txt', 'w')
# f.write(model_script)
# f.close()

In [29]:
model_script = """
model{
    b0 ~ dunif(-100, 100)
    phi ~ dunif(-100, 100)
    gamma ~ dunif(.0001, 10)
    omega ~ dunif(.0001, 10)

    for (c in 1:CCONVERSATION){
        conversation_sigma[c] ~ dnorm(omega, 1)
    }

    for (p in 1:KSPEAKER){
        speaker_sigma[p] ~ dnorm(gamma, 1)
    }

    for (r in 1:ROWS){
        # mu[r] <- (survey_var[r] * phi)
        # sigma[r] <- conversation_sigma[conversation[r]] + speaker_sigma[speaker[r]]
        slope[r] ~ dnorm(b0 + (survey_var[r] * phi), 1/pow(conversation_sigma[conversation[r]] + speaker_sigma[speaker[r]],2))
    }
}
"""
f = open('model.txt', 'w')
f.write(model_script)
f.close()


### Running JAGS model in R

In [26]:
results['other'] = (~results['self']).astype(int)
results['self'] = results['self'].astype(int)
print(results[['self', 'other']].head(n=2))

   self  other
0     1      0
1     1      0


In [27]:
results[['speaker_', 'delta_slope', 'conversation_id_', var]].head()

,speaker_,delta_slope,conversation_id_,developed_joint_perspective_sr2
0,1,0.622325,1,7.0
1,1,0.146468,1,7.0
2,2,2.437830,1,7.0
3,1,0.449710,1,7.0
4,1,0.967535,1,7.0


In [30]:
sel_2 = ~results[var].isna()

# model_ = str(model_script)
# df_ = results[['delta_slope', 'conversation_id_', 'speaker_', var]].loc[sel_2].copy()
# df_.index=range(len(df_))

with localconverter(default_converter + pandas2ri.converter):
    df_ = conversion.py2rpy(results[['delta_slope', 'conversation_id_', 'speaker_', var]].loc[sel_2])

ro.globalenv['df'] = df_

ro.r("""
library(R2jags)
print(colnames(df))
survey_var <- df${}
slope <- df$delta_slope
speaker <- df$speaker_
conversation <- df$conversation_id_
KSPEAKER <- max(speaker)
CCONVERSATION <- max(conversation)
print(KSPEAKER)
print(CCONVERSATION)
ROWS <- length(slope)
# print(KSPEAKER)
# print(CCONVERSATION)
# print(ROWS)

data <- list('survey_var', 'slope', 'speaker', 'conversation', 'CCONVERSATION', 'KSPEAKER', 'ROWS')

myinits <- list(
    list(
        gamma=1.01,
        omega=1.01,
        b0=0.1,
        phi=0.1
    )
)

parameters <- c('b0', 'omega', 'gamma', 'phi')

samples <- jags(data, inits=myinits, parameters, model.file='model.txt', n.chains=1, n.iter=1100, n.burnin=1000, DIC=T, progress.bar="text")

gamma <- samples$BUGSoutput$sims.list$gamma
# sigma <- samples$BUGSoutput$sims.list$sigma
b0 <- samples$BUGSoutput$sims.list$b0
# self_b0 <- samples$BUGSoutput$sims.list$self_b0
# other_b0 <- samples$BUGSoutput$sims.list$other_b0
phi <- samples$BUGSoutput$sims.list$phi
omega <- samples$BUGSoutput$sims.list$omega
save(b0, phi, omega, file='{}/{}.RData')
""".format(var,
            PARAMETER_ESTIMATES_PATH, var
           ))

R callback write-console: Loading required package: rjags
  
R callback write-console: Loading required package: coda
  
R callback write-console: Linked to JAGS 4.3.2
  
R callback write-console: Loaded modules: basemod,bugs
  
R callback write-console: 
Attaching package: ‘R2jags’

  
R callback write-console: The following object is masked from ‘package:coda’:

    traceplot

  


[1] "delta_slope"                     "conversation_id_"               
[3] "speaker_"                        "developed_joint_perspective_sr2"
[1] 1455
[1] 1655


R callback write-console: module glm loaded
  


Compiling model graph
   Resolving undeclared variables
   Allocating nodes
Graph information:
   Observed stochastic nodes: 382049
   Unobserved stochastic nodes: 3114
   Total graph size: 1541095

Initializing model

  |**************************************************| 100%
  |**************************************************| 100%


## Reporting

In [31]:
min_val, median_val, max_val, modal_val = 1., 5., 7., 9.

##### If importing from a checkpoint

In [32]:
from pyreadr import read_r

In [33]:
data = read_r(
    os.path.join(PARAMETER_ESTIMATES_PATH, '{}.RData').format(var)
)

##### If pulling directly from the results above

In [ ]:
data = dict()
data['beta'] = pd.DataFrame(ro.r('beta'))
data['b0'] = pd.DataFrame(ro.r('b0'))
data['phi'] = pd.DataFrame(ro.r('phi'))

### Statistical analyses/Visualizations

In [34]:
import plotly.figure_factory as ff

In [35]:
from shared.wald_test import *

test = wald(
    np.concat([
        data['b0'].values,
        data['phi'].values
    ], axis=1)
)

#### $\beta$

In [ ]:
test.test(np.array([[0,0,0,1]]))

Testing for minimum parameter estimate + $\beta_{0,other}$ difference from just $\beta_{0,other}$.

In [ ]:
test.test(np.array([[0, 1, 0, 1]]), data['other_b0'].mean().item())

Testing for median parameter estimate + $\beta_{0,other}$ difference from just $\beta_{0,other}$.

In [ ]:
test.test(np.array([[0, 1, 0, modal_val]]), data['other_b0'].mean().item())

In [ ]:
fig = ff.create_distplot(
    hist_data=[data['beta'].values.reshape(-1)],
    group_labels=['β'],
    show_hist=False,
    colors=['#354E56']
)
fig.show()

In [ ]:
fig.write_html(os.path.join(TABLES_FIGS_PATH, '{}-beta.html'.format(var)))

#### $\phi$

In [36]:
test.test(np.array([[0,1]]))

(np.float64(37.56896125803706), 1, np.float64(8.823747288388972e-10))

Testing for minimum parameter estimate + $\beta_{0,self}$ difference from just $\beta_{0,self}$.

In [ ]:
test.test(np.array([[1, 0, 1, 0]]), data['self_b0'].mean().item())

Testing for median parameter estimate + $\beta_{0,self}$ difference from just $\beta_{0,self}$.

In [ ]:
test.test(np.array([[1, 0, modal_val, 0]]), data['self_b0'].mean().item())

In [37]:
fig = ff.create_distplot(
    hist_data=[data['phi'].values.reshape(-1)],
    group_labels=['ɸ'],
    show_hist=False,
    colors=['#43572E']
)
fig.show()

In [ ]:
fig.write_html(os.path.join(TABLES_FIGS_PATH, '{}-phi.html'.format(var)))

#### Model range

In [39]:
test.test(np.array([[1,1]]))

(np.float64(10336745.518359318), 1, np.float64(0.0))

In [40]:
fig = ff.create_distplot(
    # hist_data=[
    #     # self values
    #     ((data['b0'].values.reshape(-1) + (7*(data['phi'].values.reshape(-1) + data['beta'].values.reshape(-1))))),
    #     ((data['b0'].values.reshape(-1) + (5*(data['phi'].values.reshape(-1) + data['beta'].values.reshape(-1))))),
    #     ((data['b0'].values.reshape(-1) + (1*(data['phi'].values.reshape(-1) + data['beta'].values.reshape(-1))))),
    #
    #     # other values
    #     ((data['b0'].values.reshape(-1) + (7*(data['beta'].values.reshape(-1))))),
    #     ((data['b0'].values.reshape(-1) + (5*(data['beta'].values.reshape(-1))))),
    #     ((data['b0'].values.reshape(-1) + (1*(data['beta'].values.reshape(-1))))),
    # ],
    hist_data=[
        # self values
        data['b0'].values.reshape(-1) + (max_val*data['phi'].values.reshape(-1)),
        data['b0'].values.reshape(-1) + (median_val*data['phi'].values.reshape(-1)),
        data['b0'].values.reshape(-1) + (min_val*data['phi'].values.reshape(-1)),

    ],
    group_labels=[
        'maximum', 'median', 'min',
    ],
    show_hist=False,
    colors=[
        # '#FF9F1C', '#FFBF69', '#CB997E',
        '#023E8A', '#0077B6', '#90E0EF',
    ]
)

# fig.add_vline(x=data['b0'].values.mean(), line_color='red', line_dash='dash', name='b0', showlegend=True, annotation=dict(hovertext=data['b0'].values.mean(),text=' ', ))

# fig.add_vline(x=data['b0'].values.mean(), line_color='#FF9F1C', line_dash='dash', name='b0', showlegend=True, annotation=dict(hovertext=data['b0'].values.mean(),text=' ', ))

fig.add_vline(x=data['b0'].values.mean(), line_color='#023E8A', line_dash='dash', name='b0', showlegend=True, annotation=dict(hovertext=data['b0'].values.mean(),text=' ', ))

fig.show()

In [ ]:
fig.write_html(os.path.join(TABLES_FIGS_PATH, '{}-ranges.html'.format(var)))